# Verificacion de columnas en `datos_crudos`

Este notebook permite:

1. Listar archivos y contar columnas por periodo.
2. Detectar esquemas iguales y diferentes.
3. Calcular columnas comunes entre todos los archivos.
4. Recalcular columnas comunes excluyendo `Examen_Saber_11_20141.txt`.
5. Revisar posibles nombres equivalentes (renombres o variables relacionadas).

In [1]:
from pathlib import Path
from collections import defaultdict
import difflib

# Ajusta esta ruta si moviste la carpeta
base = Path("../datos_crudos")
files = sorted(base.glob("*.txt"))

print(f"Total archivos encontrados: {len(files)}")
for f in files:
    print("-", f.name)

def read_header_columns(path: Path):
    with path.open("r", encoding="utf-8", errors="replace") as fh:
        header = fh.readline().strip("\n\r")
    return header.split(";") if header else []

headers = {f.name: read_header_columns(f) for f in files}

print("\nConteo de columnas por archivo:")
for name, cols in headers.items():
    print(f"{name}: {len(cols)}")

Total archivos encontrados: 22
- Examen_Saber_11_20141.txt
- Examen_Saber_11_20142.txt
- Examen_Saber_11_20151.txt
- Examen_Saber_11_20152.txt
- Examen_Saber_11_20161.txt
- Examen_Saber_11_20162.txt
- Examen_Saber_11_20171.txt
- Examen_Saber_11_20172.txt
- Examen_Saber_11_20181.txt
- Examen_Saber_11_20182.txt
- Examen_Saber_11_20191.txt
- Examen_Saber_11_20192.txt
- Examen_Saber_11_20201.txt
- Examen_Saber_11_20202.txt
- Examen_Saber_11_20211.txt
- Examen_Saber_11_20212.txt
- Examen_Saber_11_20221.txt
- Examen_Saber_11_20222.txt
- Examen_Saber_11_20231.txt
- Examen_Saber_11_20232.txt
- Examen_Saber_11_20241.txt
- Examen_Saber_11_20242.txt

Conteo de columnas por archivo:
Examen_Saber_11_20141.txt: 138
Examen_Saber_11_20142.txt: 57
Examen_Saber_11_20151.txt: 58
Examen_Saber_11_20152.txt: 59
Examen_Saber_11_20161.txt: 72
Examen_Saber_11_20162.txt: 72
Examen_Saber_11_20171.txt: 85
Examen_Saber_11_20172.txt: 85
Examen_Saber_11_20181.txt: 85
Examen_Saber_11_20182.txt: 85
Examen_Saber_11_201

In [2]:
# 1) Agrupar por esquema exacto (mismo set y mismo orden)
schema_groups = defaultdict(list)
for name, cols in headers.items():
    schema_groups[tuple(cols)].append(name)

print(f"Esquemas distintos encontrados: {len(schema_groups)}\n")
for i, (schema, group_files) in enumerate(
    sorted(schema_groups.items(), key=lambda kv: (len(kv[1]), kv[1][0]), reverse=True),
    start=1,
):
    print(f"G{i}: {len(group_files)} archivo(s), {len(schema)} columna(s)")
    print("   ", ", ".join(group_files))

# 2) Diferencias consecutivas (por nombre de archivo)
ordered_names = sorted(headers)
print("\nCambios de columnas entre archivos consecutivos:")
for a, b in zip(ordered_names, ordered_names[1:]):
    sa, sb = set(headers[a]), set(headers[b])
    added = sorted(sb - sa)
    removed = sorted(sa - sb)
    if added or removed:
        print(f"\n{a} -> {b}")
        if added:
            print("  +", ", ".join(added))
        if removed:
            print("  -", ", ".join(removed))

Esquemas distintos encontrados: 10

G1: 6 archivo(s), 84 columna(s)
    Examen_Saber_11_20211.txt, Examen_Saber_11_20221.txt, Examen_Saber_11_20222.txt, Examen_Saber_11_20231.txt, Examen_Saber_11_20241.txt, Examen_Saber_11_20242.txt
G2: 5 archivo(s), 85 columna(s)
    Examen_Saber_11_20182.txt, Examen_Saber_11_20191.txt, Examen_Saber_11_20192.txt, Examen_Saber_11_20201.txt, Examen_Saber_11_20212.txt
G3: 3 archivo(s), 85 columna(s)
    Examen_Saber_11_20171.txt, Examen_Saber_11_20172.txt, Examen_Saber_11_20181.txt
G4: 2 archivo(s), 72 columna(s)
    Examen_Saber_11_20161.txt, Examen_Saber_11_20162.txt
G5: 1 archivo(s), 83 columna(s)
    Examen_Saber_11_20232.txt
G6: 1 archivo(s), 84 columna(s)
    Examen_Saber_11_20202.txt
G7: 1 archivo(s), 59 columna(s)
    Examen_Saber_11_20152.txt
G8: 1 archivo(s), 58 columna(s)
    Examen_Saber_11_20151.txt
G9: 1 archivo(s), 57 columna(s)
    Examen_Saber_11_20142.txt
G10: 1 archivo(s), 138 columna(s)
    Examen_Saber_11_20141.txt

Cambios de column

In [3]:
# 3) Columnas comunes en TODOS los archivos
all_sets = [set(cols) for cols in headers.values()]
common_all = set.intersection(*all_sets)
print(f"Columnas comunes (todos los archivos): {len(common_all)}")
print(sorted(common_all))

# 4) Columnas comunes excluyendo 2014-1
excluded = "Examen_Saber_11_20141.txt"
headers_wo_20141 = {k: v for k, v in headers.items() if k != excluded}
common_wo_20141 = set.intersection(*[set(v) for v in headers_wo_20141.values()])

print(f"\nColumnas comunes (excluyendo {excluded}): {len(common_wo_20141)}")
for c in sorted(common_wo_20141):
    print("-", c)

Columnas comunes (todos los archivos): 46
['cole_area_ubicacion', 'cole_bilingue', 'cole_calendario', 'cole_caracter', 'cole_cod_dane_establecimiento', 'cole_cod_dane_sede', 'cole_cod_depto_ubicacion', 'cole_cod_mcpio_ubicacion', 'cole_codigo_icfes', 'cole_depto_ubicacion', 'cole_genero', 'cole_jornada', 'cole_mcpio_ubicacion', 'cole_naturaleza', 'cole_nombre_establecimiento', 'cole_nombre_sede', 'cole_sede_principal', 'desemp_ingles', 'estu_cod_depto_presentacion', 'estu_cod_mcpio_presentacion', 'estu_cod_reside_depto', 'estu_cod_reside_mcpio', 'estu_consecutivo', 'estu_depto_presentacion', 'estu_depto_reside', 'estu_estudiante', 'estu_fechanacimiento', 'estu_genero', 'estu_mcpio_presentacion', 'estu_mcpio_reside', 'estu_nacionalidad', 'estu_pais_reside', 'estu_privado_libertad', 'estu_tipodocumento', 'fami_cuartoshogar', 'fami_educacionmadre', 'fami_educacionpadre', 'fami_estratovivienda', 'fami_personashogar', 'fami_tieneautomovil', 'fami_tienecomputador', 'fami_tieneinternet', 'fam

In [4]:
# 5) Posibles columnas con nombre diferente pero significado relacionado
all_columns = sorted(set().union(*[set(v) for v in headers.values()]))

# Candidatos por similitud textual
pairs = []
for i, a in enumerate(all_columns):
    for b in all_columns[i+1:]:
        ratio = difflib.SequenceMatcher(None, a, b).ratio()
        if ratio >= 0.84 and a != b:
            pairs.append((ratio, a, b))

print("Posibles pares similares (similitud >= 0.84):")
for ratio, a, b in sorted(pairs, reverse=True)[:80]:
    print(f"{ratio:.2f} | {a}  <->  {b}")

# Recomendados para revisar manualmente (basado en hallazgos previos)
manual_check = [
    ("estu_pilopaga", "estu_generacione"),
    ("estu_etnia", "estu_tieneetnia"),
    ("fami_tieneserviciotv", "fami_tiene_serviciotv"),
]
print("\nRevision dirigida:")
for a, b in manual_check:
    present_a = [f for f, cols in headers.items() if a in cols]
    present_b = [f for f, cols in headers.items() if b in cols]
    print(f"\n{a} vs {b}")
    print(f"- aparece {len(present_a)} archivo(s):", present_a)
    print(f"- aparece {len(present_b)} archivo(s):", present_b)
    print(f"- solapan en {len(set(present_a) & set(present_b))} archivo(s)")

Posibles pares similares (similitud >= 0.84):
0.98 | fami_tiene_serviciotv  <->  fami_tieneserviciotv
0.97 | estu_inse_individual  <->  estu_nse_individual
0.95 | fami_trabajolabormadre  <->  fami_trabajolaborpadre
0.95 | fami_ocupacionmadre  <->  fami_ocupacionpadre
0.95 | fami_educacionmadre  <->  fami_educacionpadre
0.92 | estu_cod_mcpio_presentacion  <->  estu_mcpio_presentacion
0.92 | estu_cod_depto_presentacion  <->  estu_depto_presentacion
0.91 | estu_cod_programadeseado  <->  estu_programadeseado
0.91 | estu_cod_mcpioiesdeseada  <->  estu_mcpioiesdeseada
0.91 | cole_cod_mcpio_ubicacion  <->  cole_mcpio_ubicacion
0.91 | cole_cod_depto_ubicacion  <->  cole_depto_ubicacion
0.89 | punt_sociales_ciudadanas  <->  recaf_punt_sociales_ciudadanas
0.89 | estu_reproboseptimo  <->  estu_reprobosexto
0.89 | estu_cod_depto_presentacion  <->  estu_cod_mcpio_presentacion
0.88 | estu_cod_iesdeseada  <->  estu_cod_mcpioiesdeseada
0.88 | estu_cod_iesdeseada  <->  estu_iesdeseada
0.88 | estu_anote